# Scenario 6: Structured Data Extraction

**Domains tested:** Prompt Engineering (D4 -- 20%), Context Management (D5 -- 15%)

---

## Scenario Context

You are the data engineering lead at **MedInsure**, a health insurance company. Your team is building an automated pipeline to extract structured information from unstructured medical documents: doctor's notes, insurance claims, referral letters, and lab reports. The extracted data feeds into a claims processing system.

**Production requirements:**
- Extract structured fields (patient name, diagnosis codes, medication, dosage, dates) from free-text documents
- JSON schema validation on all extracted data -- nullable fields for legitimately missing information
- Validation-retry loop: if extraction fails schema validation, retry with the validation errors as context
- Few-shot examples for varied document formats (handwritten note transcriptions, typed reports, lab results)
- Batch processing for nightly intake of 10,000+ documents
- Confidence scoring with human review routing for low-confidence extractions

**Critical business constraints:**
- Incorrect medication or dosage extraction can lead to claim denials or patient safety issues
- Missing fields must be explicitly marked as null, not guessed
- Documents with ambiguous or conflicting information must be routed to human reviewers

---

In [ ]:
import anthropic
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

---

### Question 1 (D4)

Your extraction pipeline uses a text prompt that says: "Extract patient information from the document and return it as JSON." The extracted JSON frequently has missing fields -- sometimes the patient's date of birth is omitted even when it appears in the document.

What is the most reliable fix?

- **A)** Use tool_use with a JSON schema that defines every required field, including nullable fields for legitimately missing information -- this forces the model to produce all fields or explicitly return null
- **B)** Add "Include all fields" to the prompt
- **C)** Post-process the JSON to add missing fields with default values
- **D)** Use a larger model with better extraction capabilities

In [ ]:
q1 = {
    "correct": "A",
    "explanation": (
        "Tool use with a complete JSON schema forces the model to produce every field defined in "
        "the schema. For fields that are genuinely missing from the document, the schema should "
        "allow nullable values so the model returns null explicitly rather than omitting the field. "
        "This is structurally guaranteed by the API, unlike text-based JSON generation. "
        "Option B is vague -- 'all fields' doesn't define what fields exist. Option C adds "
        "fake data. Option D doesn't address the structural issue. This is a core Domain 4 "
        "concept: tool_use schemas for reliable structured extraction."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D4 + D5)

Your extraction pipeline processes three document formats: typed medical reports, handwritten note transcriptions, and lab results. The model performs well on typed reports but poorly on handwritten transcriptions because the formatting is inconsistent.

What is the most effective improvement?

- **A)** Use a different model for handwritten transcriptions
- **B)** Pre-process handwritten transcriptions to normalize formatting
- **C)** Add "Handle messy formatting" to the prompt
- **D)** Add few-shot examples showing each document format with its correct extraction, placed as synthetic conversation turns before the actual extraction request

In [ ]:
q2 = {
    "correct": "D",
    "explanation": (
        "Few-shot examples as synthetic conversation turns (user provides document, assistant "
        "provides correct extraction) teach the model to handle varied formats by example. "
        "Include at least one example of each document format. The model learns the mapping "
        "from messy input to clean output. Option A may not be necessary if examples solve the "
        "problem. Option B adds complexity and may lose information. Option C is vague. "
        "This tests Domain 4 (few-shot examples) and Domain 5 (effective context design for "
        "varied inputs)."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D4)

The extracted JSON sometimes fails validation because the model returns `"date_of_birth": "unknown"` instead of `"date_of_birth": null` when the date isn't in the document. Your schema requires dates to be ISO format strings or null.

What is the best fix?

- **A)** Add a post-processing step to convert "unknown" to null
- **B)** In the tool schema, define the field as `{"type": ["string", "null"]}` with a description that says: "ISO 8601 date string, or null if not present in the document. Never use placeholder strings like 'unknown' or 'N/A'"
- **C)** Remove the validation step
- **D)** Use regex to validate dates before passing to the model

In [ ]:
q3 = {
    "correct": "B",
    "explanation": (
        "The tool schema should explicitly allow null as a type alongside the expected format, "
        "with a clear description of what constitutes valid values. By stating 'Never use "
        "placeholder strings' in the description, you directly address the model's tendency to "
        "use placeholder values instead of null. Option A is a workaround that doesn't prevent "
        "the issue. Option C removes a critical safety check. Option D validates before "
        "extraction, not after. This tests Domain 4: designing tool schemas with nullable "
        "fields and clear value constraints."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D4 + D5)

Your extraction pipeline has a validation-retry loop. When extraction fails validation, it retries with the validation errors as context. After implementing this, you notice that the second attempt usually succeeds but sometimes introduces NEW errors while fixing the original ones.

What is the root cause?

- **A)** The retry prompt passes the validation errors but does not include the original extracted data -- the model re-extracts from scratch and may miss fields it got right the first time
- **B)** The model is not deterministic
- **C)** The validation is too strict
- **D)** The retry should use a different model

In [ ]:
q4 = {
    "correct": "A",
    "explanation": (
        "The retry prompt should include BOTH the validation errors AND the previous extraction "
        "result. This way, the model can fix the specific errors while preserving the correct "
        "fields. Without the previous result, the model re-extracts everything from scratch, "
        "which means fields that were correct before might be extracted differently. The fix: "
        "'Here is your previous extraction: [JSON]. These fields failed validation: [errors]. "
        "Fix ONLY the failed fields, keep everything else the same.' "
        "This tests Domain 4 (retry prompt design) and Domain 5 (passing prior results as "
        "context for iterative improvement)."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D4)

You need to process 10,000 medical documents overnight. Each document requires a separate API call for extraction. What is the most cost-effective approach?

- **A)** Run 10,000 sequential API calls
- **B)** Run 100 parallel API calls with rate limiting
- **C)** Concatenate all documents into a single API call
- **D)** Use the Anthropic Batch API to submit all 10,000 requests, getting results within 24 hours at 50% cost

In [ ]:
q5 = {
    "correct": "D",
    "explanation": (
        "The Batch API is designed for exactly this use case: high-volume, non-time-sensitive "
        "processing at 50% cost. 10,000 documents overnight is a perfect fit. Option A is slow. "
        "Option B works but costs 2x more than the Batch API. Option C would exceed context "
        "limits and mix extractions across documents. This tests Domain 4: knowing when to use "
        "the Batch API for cost-effective processing."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D4 + D5)

Your extraction system includes confidence scoring. When the model extracts a medication name, it also provides a confidence score (high/medium/low). Documents with low-confidence extractions are routed to human reviewers. But you notice the model always reports "high" confidence, even for ambiguous documents.

How do you get more calibrated confidence scores?

- **A)** Ask the model to "be honest about confidence"
- **B)** Define explicit confidence criteria in the schema: "high = exact match in document, medium = inferred from context, low = ambiguous/multiple possible values" and require the model to cite the exact text span that supports each extraction
- **C)** Use log probabilities to estimate confidence
- **D)** Run extraction 3 times and use agreement rate as confidence

In [ ]:
q6 = {
    "correct": "B",
    "explanation": (
        "Explicit confidence criteria with required evidence (text span citation) force the model "
        "to justify its confidence level. Without criteria, the model defaults to 'high' because "
        "it has no calibration framework. By requiring the exact text span, you can verify whether "
        "the confidence is justified. Option A is vague. Option C (log probs) measures token-level "
        "certainty, not semantic extraction confidence. Option D (agreement voting) works but "
        "costs 3x more. This tests Domain 4 (explicit criteria for subjective assessments) and "
        "Domain 5 (grounding confidence in source evidence)."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

### Question 7 (D5)

A medical document contains two sections that describe medications differently:

> **Current Medications:** Metformin 500mg twice daily
> 
> **Discharge Instructions:** Continue Metformin 1000mg once daily

The model extracts `Metformin 500mg twice daily` and marks confidence as "high." This is problematic because the document contains conflicting dosage information.

What is the correct system behavior?

- **A)** Extract the most recent information (discharge instructions)
- **B)** Average the dosages
- **C)** Extract both values, flag the conflict, set confidence to "low", and route to human review
- **D)** Extract neither and mark the field as null

In [ ]:
q7 = {
    "correct": "C",
    "explanation": (
        "Conflicting information in medical documents MUST be flagged, not silently resolved. "
        "The extraction should include both values, note the conflict, set confidence to low, "
        "and route to human review. In healthcare, silently choosing one value over another "
        "can have patient safety consequences. Option A makes assumptions about which section "
        "is authoritative. Option B is nonsensical for medications. Option D loses both data "
        "points. This tests Domain 5: handling conflicting information in extraction context."
    )
}
print(f"Correct: {q7['correct']}")
print(f"\nExplanation: {q7['explanation']}")

---

## Part 2: Hands-On Build

Build the structured data extraction pipeline described in the scenario.

---

### Step 1: Define the extraction schema as a tool

In [ ]:
EXTRACTION_TOOL = {
    "name": "extract_medical_data",
    "description": (
        "Extract structured medical information from a document. "
        "Use null for fields not present in the document. "
        "Never use placeholder strings like 'unknown' or 'N/A'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "patient": {
                "type": "object",
                "properties": {
                    "name": {"type": ["string", "null"], "description": "Patient full name, or null if not in document"},
                    "date_of_birth": {"type": ["string", "null"], "description": "ISO 8601 date (YYYY-MM-DD), or null if not in document"},
                    "patient_id": {"type": ["string", "null"], "description": "Patient/MRN identifier, or null if not in document"}
                },
                "required": ["name", "date_of_birth", "patient_id"]
            },
            "diagnoses": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "icd_code": {"type": ["string", "null"], "description": "ICD-10 code if mentioned, or null"},
                        "confidence": {"type": "string", "enum": ["high", "medium", "low"],
                                       "description": "high=exact text match, medium=inferred from context, low=ambiguous"},
                        "source_text": {"type": "string", "description": "Exact text from document supporting this extraction"}
                    },
                    "required": ["description", "icd_code", "confidence", "source_text"]
                }
            },
            "medications": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "dosage": {"type": ["string", "null"], "description": "Dosage with units, or null if not specified"},
                        "frequency": {"type": ["string", "null"], "description": "How often taken, or null if not specified"},
                        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                        "source_text": {"type": "string"}
                    },
                    "required": ["name", "dosage", "frequency", "confidence", "source_text"]
                }
            },
            "conflicts": {
                "type": "array",
                "description": "Conflicting information found in the document",
                "items": {
                    "type": "object",
                    "properties": {
                        "field": {"type": "string", "description": "Which field has conflicting values"},
                        "values": {"type": "array", "items": {"type": "string"}, "description": "The conflicting values found"},
                        "source_sections": {"type": "array", "items": {"type": "string"}, "description": "Which document sections contain each value"}
                    },
                    "required": ["field", "values", "source_sections"]
                }
            },
            "overall_confidence": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Overall extraction confidence. low if any conflicts exist or key fields are ambiguous."
            },
            "requires_human_review": {
                "type": "boolean",
                "description": "True if conflicts, low confidence, or ambiguous medical information detected"
            }
        },
        "required": ["patient", "diagnoses", "medications", "conflicts",
                     "overall_confidence", "requires_human_review"]
    }
}

print("Extraction tool schema defined.")
print(f"Required fields: {EXTRACTION_TOOL['input_schema']['required']}")

### Step 2: Define sample documents

In [ ]:
DOCUMENTS = {
    "typed_report": {
        "format": "Typed Medical Report",
        "content": """
PATIENT: Sarah Johnson
DOB: 1985-03-15
MRN: MRN-2024-0892

CHIEF COMPLAINT: Persistent cough for 3 weeks

ASSESSMENT:
1. Acute bronchitis (J20.9)
2. Mild hypertension, well-controlled (I10)

MEDICATIONS:
- Amoxicillin 500mg three times daily for 10 days
- Guaifenesin 400mg every 4 hours as needed
- Continue Lisinopril 10mg once daily (existing medication)

FOLLOW UP: 2 weeks if symptoms persist
"""
    },
    "handwritten_transcription": {
        "format": "Handwritten Note Transcription",
        "content": """
[Transcribed from handwritten note - some words unclear]

pt: Michael Chen  dob: 7/22/1978 or possibly 7/22/1979 (unclear handwriting)

dx: type 2 DM, A1C 8.2... also possible peripheral neuropathy
   (note: could not clearly read second diagnosis, appears to say "neuropathy")

rx: metformin 500mg bid
    ...something that looks like gabapentin [?] 300mg tid

f/u 3 months, check A1C
"""
    },
    "conflicting_document": {
        "format": "Discharge Summary with Conflicts",
        "content": """
PATIENT: Robert Williams
DOB: 1962-11-08
MRN: MRN-2024-1547

ADMISSION DIAGNOSIS: Acute myocardial infarction (I21.0)

CURRENT MEDICATIONS (from admission):
- Metformin 500mg twice daily
- Atorvastatin 20mg once daily
- Aspirin 81mg once daily

DISCHARGE MEDICATIONS:
- Metformin 1000mg once daily (dosage changed per endocrinology consult)
- Atorvastatin 40mg once daily (increased per cardiology)
- Aspirin 325mg once daily (changed per cardiology)
- Clopidogrel 75mg once daily (new)
- Metoprolol 25mg twice daily (new)

NOTE: Please follow discharge medication list. Admission medications listed for reference only.
"""
    },
    "lab_report": {
        "format": "Lab Results",
        "content": """
LABORATORY REPORT
Patient: Emily Torres
DOB: 1990-06-20
Collection Date: 2024-03-15

COMPLETE METABOLIC PANEL:
  Glucose: 245 mg/dL (H) [Reference: 70-100]
  HbA1c: 9.1% (H) [Reference: <5.7%]
  Creatinine: 1.8 mg/dL (H) [Reference: 0.6-1.2]
  BUN: 28 mg/dL (H) [Reference: 7-20]
  eGFR: 42 mL/min [Reference: >60]

INTERPRETATION:
Uncontrolled diabetes mellitus with early stage chronic kidney disease (Stage 3a).
Recommend endocrinology and nephrology referrals.
"""
    }
}

print(f"Sample documents loaded: {len(DOCUMENTS)} documents")
for key, doc in DOCUMENTS.items():
    print(f"  - {key}: {doc['format']}")

### Step 3: Build the extraction function with few-shot examples

In [ ]:
# Few-shot example as synthetic conversation turn
FEW_SHOT_EXAMPLE_DOC = """
PATIENT: Jane Doe
DOB: 1975-08-30

Assessment: Type 2 diabetes (E11.9), well-controlled
Medications: Metformin 850mg twice daily
"""

FEW_SHOT_EXAMPLE_EXTRACTION = {
    "patient": {
        "name": "Jane Doe",
        "date_of_birth": "1975-08-30",
        "patient_id": None
    },
    "diagnoses": [
        {
            "description": "Type 2 diabetes mellitus",
            "icd_code": "E11.9",
            "confidence": "high",
            "source_text": "Type 2 diabetes (E11.9), well-controlled"
        }
    ],
    "medications": [
        {
            "name": "Metformin",
            "dosage": "850mg",
            "frequency": "twice daily",
            "confidence": "high",
            "source_text": "Metformin 850mg twice daily"
        }
    ],
    "conflicts": [],
    "overall_confidence": "high",
    "requires_human_review": False
}


def extract_medical_data(document: str, use_few_shot: bool = True) -> dict:
    """Extract structured medical data using tool_use with optional few-shot examples."""

    system = """You are a medical document extraction system. Extract structured data from medical documents.

RULES:
1. Extract ONLY information explicitly stated in the document.
2. Use null for fields not present -- NEVER guess or use placeholders like "unknown".
3. For each extraction, cite the exact source text from the document.
4. Confidence levels:
   - high: exact, unambiguous text in the document
   - medium: inferred from context (e.g., abbreviations, implied information)
   - low: ambiguous, unclear handwriting, or multiple possible interpretations
5. Flag ALL conflicting information -- do not silently resolve conflicts.
6. Set requires_human_review = true if:
   - Any conflicts exist
   - Any medication has low confidence
   - Key patient identifiers are ambiguous
   - The document contains unclear or potentially misread text"""

    messages = []

    # Add few-shot example as synthetic conversation turn
    if use_few_shot:
        messages.append({
            "role": "user",
            "content": f"Extract medical data from this document:\n{FEW_SHOT_EXAMPLE_DOC}"
        })
        messages.append({
            "role": "assistant",
            "content": [
                {
                    "type": "tool_use",
                    "id": "example_1",
                    "name": "extract_medical_data",
                    "input": FEW_SHOT_EXAMPLE_EXTRACTION
                }
            ]
        })
        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": "example_1",
                    "content": "Extraction recorded successfully."
                }
            ]
        })

    # Add the actual document
    messages.append({
        "role": "user",
        "content": f"Extract medical data from this document:\n{document}"
    })

    response = client.messages.create(
        model=MODEL, max_tokens=4096,
        system=system,
        tools=[EXTRACTION_TOOL],
        messages=messages
    )

    for block in response.content:
        if block.type == "tool_use" and block.name == "extract_medical_data":
            return block.input

    return {"error": "No extraction produced"}


print("Extraction function ready.")

### Step 4: Test extraction on all document types

In [ ]:
print("=== Test 1: Typed Medical Report (clean, structured) ===")
result = extract_medical_data(DOCUMENTS["typed_report"]["content"])
print(json.dumps(result, indent=2))

print(f"\nOverall confidence: {result.get('overall_confidence', 'N/A')}")
print(f"Requires human review: {result.get('requires_human_review', 'N/A')}")
print(f"Conflicts: {len(result.get('conflicts', []))}")

In [ ]:
print("=== Test 2: Handwritten Transcription (ambiguous) ===")
result = extract_medical_data(DOCUMENTS["handwritten_transcription"]["content"])
print(json.dumps(result, indent=2))

print(f"\nOverall confidence: {result.get('overall_confidence', 'N/A')}")
print(f"Requires human review: {result.get('requires_human_review', 'N/A')}")
print(f"Conflicts: {len(result.get('conflicts', []))}")

# Check: DOB should be low confidence (two possible years)
dob = result.get('patient', {}).get('date_of_birth')
print(f"\nDOB extracted: {dob}")
print("Expected: DOB should have low confidence or be flagged due to unclear handwriting")

In [ ]:
print("=== Test 3: Document with Conflicting Information ===")
result = extract_medical_data(DOCUMENTS["conflicting_document"]["content"])
print(json.dumps(result, indent=2))

print(f"\nOverall confidence: {result.get('overall_confidence', 'N/A')}")
print(f"Requires human review: {result.get('requires_human_review', 'N/A')}")
print(f"Conflicts detected: {len(result.get('conflicts', []))}")
for c in result.get('conflicts', []):
    print(f"  - {c.get('field', '?')}: {c.get('values', [])}")

In [ ]:
print("=== Test 4: Lab Report (numerical data) ===")
result = extract_medical_data(DOCUMENTS["lab_report"]["content"])
print(json.dumps(result, indent=2))

print(f"\nOverall confidence: {result.get('overall_confidence', 'N/A')}")
print(f"Requires human review: {result.get('requires_human_review', 'N/A')}")

### Step 5: Build the validation-retry loop

In [ ]:
def validate_extraction(data: dict) -> list:
    """Validate extracted data against business rules. Returns list of errors."""
    errors = []

    # Check patient fields
    patient = data.get("patient", {})
    if not patient.get("name"):
        errors.append("patient.name is missing or null")

    dob = patient.get("date_of_birth")
    if dob and dob not in [None]:
        try:
            datetime.fromisoformat(dob)
        except (ValueError, TypeError):
            errors.append(f"patient.date_of_birth '{dob}' is not valid ISO 8601 format")

    # Check medications
    for i, med in enumerate(data.get("medications", [])):
        if not med.get("name"):
            errors.append(f"medications[{i}].name is missing")
        if not med.get("source_text"):
            errors.append(f"medications[{i}].source_text is missing (required for provenance)")
        if med.get("confidence") not in ["high", "medium", "low"]:
            errors.append(f"medications[{i}].confidence must be high/medium/low")

    # Check diagnoses
    for i, dx in enumerate(data.get("diagnoses", [])):
        if not dx.get("description"):
            errors.append(f"diagnoses[{i}].description is missing")
        if not dx.get("source_text"):
            errors.append(f"diagnoses[{i}].source_text is missing (required for provenance)")

    # Business rule: conflicts should trigger human review
    if data.get("conflicts") and not data.get("requires_human_review"):
        errors.append("Conflicts detected but requires_human_review is not set to true")

    return errors


def extract_with_retry(document: str, max_retries: int = 2) -> dict:
    """Extract with validation-retry loop."""
    for attempt in range(max_retries + 1):
        if attempt == 0:
            result = extract_medical_data(document)
        else:
            # Retry with validation errors AND previous result as context
            retry_prompt = f"""Extract medical data from this document:
{document}

IMPORTANT: Your previous extraction had these validation errors:
{json.dumps(errors, indent=2)}

Your previous extraction was:
{json.dumps(result, indent=2)}

Fix ONLY the fields with errors. Keep all correct fields unchanged."""

            response = client.messages.create(
                model=MODEL, max_tokens=4096,
                system="You are a medical document extraction system. Fix the validation errors.",
                tools=[EXTRACTION_TOOL],
                messages=[{"role": "user", "content": retry_prompt}]
            )

            for block in response.content:
                if block.type == "tool_use" and block.name == "extract_medical_data":
                    result = block.input
                    break

        errors = validate_extraction(result)
        print(f"  Attempt {attempt + 1}: {len(errors)} validation errors")
        if errors:
            for e in errors:
                print(f"    - {e}")
        else:
            print(f"  Passed validation on attempt {attempt + 1}")
            return {"data": result, "attempts": attempt + 1, "valid": True}

    return {"data": result, "attempts": max_retries + 1, "valid": False,
            "remaining_errors": errors}


print("=== Validation-Retry Loop ===")
for name, doc in DOCUMENTS.items():
    print(f"\n--- {name} ({doc['format']}) ---")
    result = extract_with_retry(doc["content"])
    print(f"  Valid: {result['valid']}, Attempts: {result['attempts']}")
    if result.get('remaining_errors'):
        print(f"  Remaining errors: {result['remaining_errors']}")

---

## Part 3: Failure Injections

---

### Failure 1: No few-shot examples (poor handwriting handling)

In [ ]:
print("=== FAILURE INJECTION: No few-shot examples ===")
print("Extracting from handwritten transcription WITHOUT few-shot examples...\n")

# Without few-shot
result_no_fewshot = extract_medical_data(
    DOCUMENTS["handwritten_transcription"]["content"],
    use_few_shot=False
)

# With few-shot
result_with_fewshot = extract_medical_data(
    DOCUMENTS["handwritten_transcription"]["content"],
    use_few_shot=True
)

print("WITHOUT few-shot:")
print(f"  Human review flagged: {result_no_fewshot.get('requires_human_review', 'N/A')}")
print(f"  Confidence: {result_no_fewshot.get('overall_confidence', 'N/A')}")
print(f"  Medications: {len(result_no_fewshot.get('medications', []))}")

print("\nWITH few-shot:")
print(f"  Human review flagged: {result_with_fewshot.get('requires_human_review', 'N/A')}")
print(f"  Confidence: {result_with_fewshot.get('overall_confidence', 'N/A')}")
print(f"  Medications: {len(result_with_fewshot.get('medications', []))}")

print("\n--- DIAGNOSIS ---")
print("Few-shot examples teach the model:")
print("  - How to handle null fields (use null, not 'unknown')")
print("  - How to cite source text")
print("  - How to calibrate confidence levels")
print("  - How to handle ambiguous handwriting notation")
print("LESSON: Few-shot examples are critical for varied document formats (Domain 4).")

### Failure 2: Retry without prior result

In [ ]:
print("=== FAILURE INJECTION: Retry without prior extraction ===")
print("Simulating what happens when the retry doesn't include the previous result...\n")

# First extraction
first_result = extract_medical_data(DOCUMENTS["typed_report"]["content"])
first_meds = [m["name"] for m in first_result.get("medications", [])]
print(f"First extraction medications: {first_meds}")

# Simulate a retry WITHOUT the previous result
retry_prompt_bad = f"""Extract medical data from this document:
{DOCUMENTS['typed_report']['content']}

IMPORTANT: Fix this validation error:
- patient.date_of_birth needs ISO 8601 format

Extract all data again."""

response = client.messages.create(
    model=MODEL, max_tokens=4096,
    tools=[EXTRACTION_TOOL],
    messages=[{"role": "user", "content": retry_prompt_bad}]
)

for block in response.content:
    if block.type == "tool_use" and block.name == "extract_medical_data":
        retry_result_bad = block.input
        break

retry_meds_bad = [m["name"] for m in retry_result_bad.get("medications", [])]
print(f"Retry medications (no prior context): {retry_meds_bad}")

# Check for differences
if set(first_meds) != set(retry_meds_bad):
    print("\nMEDICATION MISMATCH between first extraction and retry!")
    print(f"  First: {first_meds}")
    print(f"  Retry: {retry_meds_bad}")
else:
    print("\nSame medications (but other fields may differ).")

print("\n--- DIAGNOSIS ---")
print("Without the previous extraction as context, the retry:")
print("  - Re-extracts from scratch (may miss previously correct fields)")
print("  - May introduce new errors while fixing the original one")
print("  - Cannot preserve correct fields reliably")
print("FIX: Always include the previous result in retry prompts (Domain 4, D5).")

### Failure 3: Silent conflict resolution

In [ ]:
print("=== FAILURE INJECTION: Prompting for silent conflict resolution ===")
print()

# Bad prompt that encourages silent resolution
bad_system = """Extract medical data from documents. Always provide a single, definitive
answer for each field. Choose the most likely value when information is ambiguous."""

response = client.messages.create(
    model=MODEL, max_tokens=4096,
    system=bad_system,
    tools=[EXTRACTION_TOOL],
    messages=[{
        "role": "user",
        "content": f"Extract:\n{DOCUMENTS['conflicting_document']['content']}"
    }]
)

for block in response.content:
    if block.type == "tool_use" and block.name == "extract_medical_data":
        silent_result = block.input
        break

print("With 'choose the most likely value' prompt:")
print(f"  Conflicts detected: {len(silent_result.get('conflicts', []))}")
print(f"  Requires human review: {silent_result.get('requires_human_review', 'N/A')}")

# Show what the correct extraction should flag
good_result = extract_medical_data(DOCUMENTS["conflicting_document"]["content"])
print(f"\nWith proper conflict detection prompt:")
print(f"  Conflicts detected: {len(good_result.get('conflicts', []))}")
print(f"  Requires human review: {good_result.get('requires_human_review', 'N/A')}")

print("\n--- DIAGNOSIS ---")
print("'Choose the most likely value' encourages the model to silently resolve conflicts.")
print("In medical data, this can lead to incorrect medication dosages reaching the")
print("claims system. The prompt must EXPLICITLY instruct the model to flag conflicts.")
print("LESSON: For safety-critical domains, always surface ambiguity rather than")
print("resolving it silently (Domain 4, D5).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| tool_use for extraction | D4 | Schema-enforced output guarantees all fields are present and typed correctly |
| Nullable fields | D4 | Use `["string", "null"]` for legitimately missing data -- never allow placeholder strings |
| Few-shot examples | D4, D5 | Synthetic conversation turns teach handling of varied document formats |
| Validation-retry loops | D4, D5 | Include BOTH validation errors AND previous result in retry context |
| Confidence criteria | D4 | Define explicit levels (high/medium/low) with required evidence citations |
| Conflict detection | D5 | Never silently resolve conflicts -- flag them and route to human review |
| Batch API | D4 | 50% cost reduction for high-volume overnight processing |
| Human review routing | D4, D5 | Low confidence + conflicts = automatic routing to human reviewers |